# Linear Regression Model Interpretability




## **1. Abstract**

The insurance risk are important point of estimation for the insurance company. The aim of the notebook is to find the factors affecting the this and creating a regression model to predict this. Interpreting the linear model using azure interpretation and dashboard.



## **2. Importing necessary libraries**

In [ ]:
import numpy as np
from matplotlib import pyplot
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pylab as plt
from sklearn.ensemble import RandomForestRegressor

## **3. Data Loading and Preprocessing**

In [ ]:
#Reading the ablone data

data = pd.read_csv('insurance.csv')
data

In [ ]:
#Seeing the data in the file 
data.head(20)

### **3.1. Encoding the sex and  smoker**

In [ ]:

data['sex'] = data.sex.map({'male':0, 'female':1})

data['smoker'] = data.smoker.map({'no':0, 'yes':1})
data.head(20)

In [ ]:
data.region.unique()

## **3.2. One Hot Encoding**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

In [ ]:
# creating instance of one-hot-encoder
enc = OneHotEncoder()
# passing bridge-types-cat column (label encoded values of bridge_types)
enc_df = pd.DataFrame(enc.fit_transform(data[['region']]).toarray())
enc_df
enc_df.columns = ['northeast','northwest','southeast','southwest'] 

enc_df.apply(np.int64)
data =data.join(enc_df)
data=data.drop(['region'],axis=1)
data

In [ ]:
data.columns

## **3.3. Noramlizing the data**

In [ ]:

# Normalizing the data in the rings column beacuse the value is too high when compared to independent variable

from sklearn import preprocessing

# Create x, where x the 'scores' column's values as floats
x = data[['age', 'sex', 'bmi', 'children', 'smoker', 'charges', 'northeast',
       'northwest', 'southeast', 'southwest']].values.astype(float)

# Create a minimum and maximum processor object
min_max_scaler = preprocessing.MinMaxScaler()


# Create an object to transform the data to fit minmax processor
x_scaled = min_max_scaler.fit_transform(x)


# Run the normalizer on the dataframe
data[['age', 'sex', 'bmi', 'children', 'smoker', 'charges', 'northeast',
       'northwest', 'southeast', 'southwest']] = pd.DataFrame(x_scaled)



In [ ]:
#looking at data head after adding dummy variables and nromalizing
data.head(20)

In [ ]:
#finding the correlation between all the features in the data
data.corr()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#plotting the heat map of the correlation
plt.figure(figsize=(20,7))
sns.heatmap(data.corr(), annot=True)

In [ ]:
#The column names 
data.columns

## **3.4. Train ,Validation and Test split**

Data is split into 3 parts 

Taining data set = 80.75%

Validation data set = 14.25%

Test data set = 5%

In [ ]:
from sklearn.model_selection import  train_test_split

X = data[ ['age', 'sex', 'bmi', 'children', 'smoker', 'northeast',
       'northwest', 'southeast', 'southwest']]

y = data['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=1)

## **4. Tarining Linear Regression model**

In [ ]:
from sklearn.model_selection import  train_test_split
from sklearn.metrics import r2_score 
from sklearn.linear_model import LinearRegression
from sklearn import datasets, linear_model
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# Create linear regression object
regr = linear_model.LinearRegression()

# Train the model using the training sets
model = regr.fit(X_train,y_train)

# Make predictions using the testing set
y_pred = regr.predict(X_train)

In [ ]:
#training Data
# The coefficients
Weights = regr.coef_
print('Coefficients: \n', regr.coef_)
# The mean squared error
print('Mean squared error: %.2f'% mean_squared_error(y_train, y_pred))
# The coefficient of determination: 1 is perfect prediction
print('Coefficient of determination: %.2f'% r2_score(y_train, y_pred))


## **5. Importing SHAP and other required libraries**

In [ ]:
from sklearn import datasets
from sklearn.ensemble import GradientBoostingRegressor

# Explainers:
# 1. SHAP Tabular Explainer
from interpret.ext.blackbox import TabularExplainer

# OR

# 2. Mimic Explainer
#from interpret.ext.blackbox import MimicExplainer
# You can use one of the following four interpretable models as a global surrogate to the black box model
from interpret.ext.glassbox import LGBMExplainableModel
from interpret.ext.glassbox import LinearExplainableModel
from interpret.ext.glassbox import SGDExplainableModel
from interpret.ext.glassbox import DecisionTreeExplainableModel

In [ ]:
data.columns
featu_na = a = np.array(['age', 'sex', 'bmi', 'children', 'smoker', 'northeast',
       'northwest', 'southeast', 'southwest'])
featu_na.shape

In [ ]:
explainer = TabularExplainer(model,X_train,features =featu_na)

In [ ]:
global_explanation = explainer.explain_global(X_test)

In [ ]:
print('ranked global importance values: {}'.format(global_explanation.get_ranked_global_values()))
# Corresponding feature names
print('ranked global importance names: {}'.format(global_explanation.get_ranked_global_names()))
# Feature ranks (based on original order of features)
print('global importance rank: {}'.format(global_explanation.global_importance_rank))

In [ ]:
# Print out a dictionary that holds the sorted feature importance names and values
print('global importance rank: {}'.format(global_explanation.get_feature_importance_dict()))

In [ ]:
# Note: Do not run this cell if using PFIExplainer, it does not support local explanations
# feature shap values for all features and all data points in the training data
print('local importance values: {}'.format(global_explanation.local_importance_values))
#Generate local explanations

In [ ]:
# You can pass a specific data point or a group of data points to the explain_local function

# E.g., Explain the first data point in the test set
local_explanation = explainer.explain_local(X_test)

# E.g., Explain the first five data points in the test set
# local_explanation_group = explainer.explain_local(x_test[0:4,:])

In [ ]:
# Sorted local feature importance information; reflects the original feature order
sorted_local_importance_names = local_explanation.get_ranked_local_names()
sorted_local_importance_values = local_explanation.get_ranked_local_values()

print('sorted local importance names: {}'.format(sorted_local_importance_names))
print('sorted local importance values: {}'.format(sorted_local_importance_values))

In [ ]:
from interpret_community.widget import ExplanationDashboard

## **6. Dashboard**

In [ ]:
ExplanationDashboard(global_explanation, model, datasetX=X_test)

## **7. Conclusion**

The dashboard helps for both local and global interpretation, perturbation helps how changing one value affects the output. This helps immensely to understand the linear model.


## **8. Refrence** 

1. https://docs.microsoft.com/en-us/azure/machine-learning/how-to-machine-learning-interpretability
2. https://www.youtube.com/watch?v=EIvMxWfsOHo


Copyright 2020 Abhishek Gargha Maheshwarappa and Kartik Kumar

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.

Copyright 2020 Abhishek Garga Maheshwarappa and Kartik Kumar



Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.